# 09 · Onsite dataset (Phase 2) and resource-demand audit — Stage 0

**Scope.** Integrity of the 15-column onsite dataset, its equivalence with the Phase 1 forecasting inputs, and how the three new
resource columns (`saline_bags_needed`, `diagnostic_kits_needed`, `hospital_beds_required`) relate to dengue cases.

**Sealed year.** 2025 is the locked test year. Every value-level analysis here uses development rows (2016–2024) through
`load_onsite_development_panel()`, which drops 2025 rows as bytes before any value is parsed. The 2025 block is checked for
structure only (row, field and key counts) and by checksum, never summarised.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import json
from fractions import Fraction
import numpy as np, pandas as pd
from dengueshield.data import COLUMNS, load_development_panel
from dengueshield.onsite_data import ONSITE_PATH, ONSITE_SHA256, RESOURCES, load_onsite_development_panel, forecast_view
pd.set_option("display.width", 170)
onsite = load_onsite_development_panel()      # checksum-verified; 2016-2024 only
phase1 = load_development_panel()             # the Phase 1 file, same seal
print("onsite development panel", onsite.shape, "| Phase 1 development panel", phase1.shape)

onsite development panel (28200, 15) | Phase 1 development panel (28200, 12)


## 1. Integrity of the development rows (2016–2024)

In [2]:
pd.Series({
    "rows": len(onsite), "columns": onsite.shape[1], "wards": onsite.ward_id.nunique(), "cities": onsite.city.nunique(),
    "weeks per ward": int(onsite.groupby("ward_id").size().unique()[0]), "first week": str(onsite.date.min().date()),
    "last development week": str(onsite.date.max().date()), "all dates are Mondays": bool((onsite.date.dt.dayofweek == 0).all()),
    "missing cells": int(onsite.isna().sum().sum()), "duplicate rows": int(onsite.duplicated().sum()),
    "duplicate (ward, week)": int(onsite.duplicated(["ward_id", "date"]).sum())}).to_frame("value")

,value
rows,28200
columns,15
wards,60
cities,5
weeks per ward,470
first week,2016-01-04
last development week,2024-12-30
all dates are Mondays,True
missing cells,0
duplicate rows,0


In [3]:
onsite.groupby("city").agg(wards=("ward_id", "nunique"), rows=("ward_id", "size")).join(
    onsite.assign(prefix=onsite.ward_id.str.split("_Ward").str[0]).groupby("city").prefix.unique().rename("ward prefix"))

,wards,rows,ward prefix
city,,,
Chattogram,10,4700,[CCC]
Dhaka North,20,9400,[DNCC]
Dhaka South,20,9400,[DSCC]
Gazipur,5,2350,[GCC]
Narayanganj,5,2350,[NCC]


In [4]:
onsite.drop(columns=["date", "city", "ward_id"]).agg(["min", "max", "mean"]).T.round(3)

,min,max,mean
population_density_sqkm,18314.00,74252.00,43157.733
water_logging_risk_index,0.22,0.94,0.579
sanitation_score,0.25,0.88,0.555
avg_temp_c,17.25,37.51,26.930
max_temp_c,23.12,40.79,31.906
min_temp_c,13.83,31.64,21.962
rainfall_mm,0.00,268.83,35.620
relative_humidity_pct,41.84,99.00,71.142
dengue_cases,0.00,41.00,2.665
saline_bags_needed,0.00,98.00,6.336


## 2. The forecasting inputs are unchanged

The 12 Phase 1 columns of the onsite file equal the Phase 1 file value by value on every development row, so the champion bundle, its replay and its validation results remain valid without retraining.

In [5]:
a, b = forecast_view(onsite).reset_index(drop=True), phase1.reset_index(drop=True)
same = {c: bool(a[c].equals(b[c])) for c in COLUMNS}
assert all(same.values())
pd.Series(same).to_frame("identical to Phase 1")

,identical to Phase 1
date,True
city,True
ward_id,True
population_density_sqkm,True
water_logging_risk_index,True
sanitation_score,True
avg_temp_c,True
max_temp_c,True
min_temp_c,True
rainfall_mm,True


## 3. Sealed 2025 block: structure only

Bytes are split, never parsed into numbers. The 12 forecasting columns are compared with the Phase 1 file's 2025 block by checksum.

In [6]:
import hashlib
def sealed_lines(path):
    return [l.rstrip(b"\r") for l in Path(path).read_bytes().split(b"\n")[1:] if l.strip() and l.split(b",", 1)[0] >= b"2025-01-01"]
sealed, sealed_p1 = sealed_lines(ONSITE_PATH), sealed_lines(ROOT / "data" / "raw" / "bangladesh_dengue_large_dataset.csv")
digest = lambda ls: hashlib.sha256(b"\n".join(b",".join(l.split(b",")[:12]) for l in ls)).hexdigest()
pd.Series({"rows": len(sealed), "field counts": sorted({len(l.split(b",")) for l in sealed}),
           "rows with an empty field": sum(1 for l in sealed if b",," in l or l.endswith(b",")),
           "weeks": len({l.split(b",", 1)[0] for l in sealed}), "wards": len({l.split(b",")[2] for l in sealed}),
           "12 forecasting columns byte-identical to Phase 1": digest(sealed) == digest(sealed_p1)}).to_frame("value")

,value
rows,3120
field counts,[15]
rows with an empty field,0
weeks,52
wards,60
12 forecasting columns byte-identical to Phase 1,True


## 4. How the resources relate to cases

Each resource is a function of the **same week's** case count alone: one value per case count, in every ward, city and season. Zero cases always means zero resources.

In [7]:
tab = onsite.groupby("dengue_cases")[RESOURCES].nunique()
print("most distinct values of a resource for one case count:", tab.max().to_dict())
print("rows with zero cases but a non-zero resource:", {r: int(((onsite.dengue_cases == 0) & (onsite[r] > 0)).sum()) for r in RESOURCES})
onsite.groupby("dengue_cases")[RESOURCES].first().T

most distinct values of a resource for one case count: {'saline_bags_needed': 1, 'diagnostic_kits_needed': 1, 'hospital_beds_required': 1}
rows with zero cases but a non-zero resource: {'saline_bags_needed': 0, 'diagnostic_kits_needed': 0, 'hospital_beds_required': 0}


dengue_cases,0,1,2,3,4,5,6,7,8,9,...,28,29,30,31,32,33,34,38,39,41
saline_bags_needed,0,2,5,7,10,12,14,17,19,21,...,67,69,72,74,76,79,81,91,93,98
diagnostic_kits_needed,0,1,2,3,5,6,7,8,9,10,...,33,34,35,36,37,38,40,44,45,48
hospital_beds_required,0,0,1,1,1,1,2,2,2,3,...,8,9,9,9,10,10,10,11,12,12


### Exact rule identification

For a candidate rule `resource = floor(a · cases + ½)` (round half up), each observed pair (c > 0, r) allows only
`a ∈ [(2r − 1)/2c, (2r + 1)/2c)`. Intersecting these intervals over all pairs, in exact rational arithmetic, gives every multiplier
consistent with the data. A non-empty intersection means the rule reproduces every row; its width is the remaining ambiguity.

In [8]:
def feasible(df, r):
    lo, hi = Fraction(-10**9), Fraction(10**9)
    for c, v in df.loc[df.dengue_cases > 0, ["dengue_cases", r]].drop_duplicates().itertuples(index=False):
        lo, hi = max(lo, Fraction(2 * int(v) - 1, 2 * int(c))), min(hi, Fraction(2 * int(v) + 1, 2 * int(c)))
    return lo, hi

def to_resource(cases, a):
    return np.floor(np.asarray(cases, dtype=float) * float(a) + 0.5).astype(int)

rule = {}
for r in RESOURCES:
    lo, hi = feasible(onsite, r)
    assert lo < hi, r
    rule[r] = {"lo": lo, "hi": hi, "a": (lo + hi) / 2}
pd.DataFrame({r: {"lower (inclusive)": float(v["lo"]), "upper (exclusive)": float(v["hi"]), "chosen: midpoint": float(v["a"]),
                  "width": float(v["hi"] - v["lo"])} for r, v in rule.items()}).T.round(6)

,lower (inclusive),upper (exclusive),chosen: midpoint,width
saline_bags_needed,2.383333,2.386364,2.384848,0.003030
diagnostic_kits_needed,1.161765,1.166667,1.164216,0.004902
hospital_beds_required,0.296875,0.300000,0.298438,0.003125


In [9]:
repro = {r: float((to_resource(onsite.dengue_cases, rule[r]["a"]) == onsite[r]).mean()) for r in RESOURCES}
assert all(v == 1.0 for v in repro.values())
print("share of development rows reproduced exactly:", repro)

share of development rows reproduced exactly: {'saline_bags_needed': 1.0, 'diagnostic_kits_needed': 1.0, 'hospital_beds_required': 1.0}


### Time invariance and past-only identification

Every season admits the same rule, and a rule identified on 2016–2018 alone reproduces 2019–2024 exactly. Converting forecasts with a rule identified strictly before the evaluated targets therefore gives the same numbers as the full-window rule: identifying the constant carries no information about future outcomes.

In [10]:
per_year = pd.DataFrame({y: {r: f"[{float(feasible(g, r)[0]):.4f}, {float(feasible(g, r)[1]):.4f})" for r in RESOURCES}
                         for y, g in onsite.groupby(onsite.date.dt.year)}).T
early, late = onsite[onsite.date.dt.year <= 2018], onsite[onsite.date.dt.year >= 2019]
early_rule = {r: sum(feasible(early, r)) / 2 for r in RESOURCES}
past_only = {r: float((to_resource(late.dengue_cases, early_rule[r]) == late[r]).mean()) for r in RESOURCES}
assert all(v == 1.0 for v in past_only.values())
print("2019-2024 reproduced by the rule identified on 2016-2018 only:", past_only)
per_year

2019-2024 reproduced by the rule identified on 2016-2018 only: {'saline_bags_needed': 1.0, 'diagnostic_kits_needed': 1.0, 'hospital_beds_required': 1.0}


,saline_bags_needed,diagnostic_kits_needed,hospital_beds_required
2016,"[2.3824, 2.3864)","[1.1618, 1.1667)","[0.2969, 0.3000)"
2017,"[2.3824, 2.3864)","[1.1591, 1.1667)","[0.2955, 0.3000)"
2018,"[2.3824, 2.3864)","[1.1591, 1.1667)","[0.2955, 0.3000)"
2019,"[2.3824, 2.3864)","[1.1607, 1.1667)","[0.2955, 0.3000)"
2020,"[2.3824, 2.3864)","[1.1591, 1.1667)","[0.2955, 0.3000)"
2021,"[2.3824, 2.3864)","[1.1591, 1.1667)","[0.2955, 0.3000)"
2022,"[2.3824, 2.3864)","[1.1591, 1.1667)","[0.2955, 0.3000)"
2023,"[2.3833, 2.3864)","[1.1607, 1.1667)","[0.2969, 0.3000)"
2024,"[2.3824, 2.3864)","[1.1591, 1.1667)","[0.2955, 0.3000)"


## 5. From predicted cases to predicted resources (leakage-safe)

The conversion is applied to **forecasts**, never to observed target-week cases or resources. The rule is monotone non-decreasing,
so quantiles map exactly: the resource median and 80%/90% ranges are the rule applied to the case median and range ends.

Evaluation on the committed out-of-time replay (2019–2024; each season forecast by a model trained only on earlier seasons), using
the rule identified on 2016–2018. Actual resources are used only to score.

In [11]:
rep = pd.read_csv(ROOT / "app_data" / "replay.csv.gz", parse_dates=["origin_date", "target_date"])
rep = rep[rep.target_date.dt.year <= 2024]
j = rep.join(onsite.set_index(["ward_id", "date"])[RESOURCES], on=["ward_id", "target_date"], how="inner")
assert len(j) == len(rep)
rows = []
for r in RESOURCES:
    a = early_rule[r]
    for h, g in j.groupby("horizon"):
        med, lo80, hi80, lo90, hi90 = (to_resource(g[q], a) for q in ["median", "lo80", "hi80", "lo90", "hi90"])
        y = g[r].to_numpy()
        rows.append({"resource": r, "horizon": int(h), "ward-weeks": len(g), "MAE": float(np.abs(med - y).mean()),
                     "bias (forecast - actual)": float((med - y).mean()), "80% range held": float(((y >= lo80) & (y <= hi80)).mean()),
                     "90% range held": float(((y >= lo90) & (y <= hi90)).mean()),
                     "cases: 80% range held": float(((g.y_true >= g.lo80) & (g.y_true <= g.hi80)).mean())})
ev = pd.DataFrame(rows)
ev.round(3)

,resource,horizon,ward-weeks,MAE,bias (forecast - actual),80% range held,90% range held,cases: 80% range held
0,saline_bags_needed,2,18060,2.634,-0.366,0.888,0.930,0.888
1,saline_bags_needed,3,17700,3.294,-0.192,0.850,0.901,0.850
2,saline_bags_needed,4,17340,3.779,0.022,0.825,0.901,0.825
3,diagnostic_kits_needed,2,18060,1.283,-0.159,0.888,0.930,0.888
4,diagnostic_kits_needed,3,17700,1.602,-0.076,0.850,0.901,0.850
5,diagnostic_kits_needed,4,17340,1.833,0.020,0.825,0.901,0.825
6,hospital_beds_required,2,18060,0.321,-0.067,0.938,0.962,0.888
7,hospital_beds_required,3,17700,0.413,-0.040,0.910,0.944,0.850
8,hospital_beds_required,4,17340,0.477,-0.010,0.891,0.941,0.825


Resource coverage is at least the case coverage (a monotone map cannot lose a covered week, and rounding can merge neighbouring counts), and the errors scale with the multipliers. The planning risk is therefore the forecast's own error, above all at outbreak peaks, where Phase 1 documented under-prediction and under-coverage.

## 6. Bed demand per ward (input to transparent capacity assumptions)

The dataset has **no bed capacity**. The weekly bed requirement per ward over 2016–2024 is shown only to inform configurable planning defaults; it is not a capacity.

In [12]:
beds = onsite.groupby(["city", "ward_id"]).hospital_beds_required.agg(p50="median", p90=lambda s: s.quantile(0.9), p99=lambda s: s.quantile(0.99), max="max")
beds.groupby("city").agg(["min", "median", "max"]).round(1)

p50              p90              p99             max           
             min median  max  min median  max  min median  max min median max
city                                                                         
Chattogram   0.0    0.0  0.0  2.0    3.0  4.0  3.0    5.2  8.0   4    7.0  11
Dhaka North  0.0    0.0  0.0  2.0    3.0  4.0  4.0    5.0  9.0   4    7.0  12
Dhaka South  0.0    0.0  0.0  1.0    3.0  4.0  3.0    5.0  7.0   4    7.0  12
Gazipur      0.0    0.0  0.0  3.0    3.0  4.0  5.0    6.3  8.3   6    8.0  10
Narayanganj  0.0    0.0  0.0  2.0    2.0  4.0  4.0    5.0  7.0   5    6.0   8

In [13]:
city_week = onsite.groupby(["city", "date"])[RESOURCES].sum()
city_week.groupby("city").agg(["median", "max"])

saline_bags_needed      diagnostic_kits_needed      hospital_beds_required     
                        median  max                 median  max                 median  max
city                                                                                       
Chattogram                 5.5  493                    2.5  243                    0.0   63
Dhaka North               12.5  914                    6.0  444                    0.0  113
Dhaka South               13.5  913                    6.0  444                    1.0  115
Gazipur                    4.0  288                    2.0  140                    0.0   37
Narayanganj                2.0  200                    1.0   98                    0.0   25

## 7. What the dataset does not contain

| Needed for | Missing in the data | Planned handling (configurable, labelled as an assumption) |
|---|---|---|
| Problem 1: allocation under scarcity | Actual regional stock of saline, kits and beds | Available stock = slider (30–100%) × the projected requirement of the pool at the chosen service level (median, upper 80% or upper 90%) |
| Problem 1: crew assignment | Number of spraying crews, crew capacity | Total crews = user input; one crew per ward-week unless the officer changes it |
| Problem 2: overflow routing | Ward bed capacity | Per-ward capacity = configurable default derived from the ward's own 2016–2024 bed demand (e.g. p90), shown as an assumption, editable; never presented as real hospital capacity |
| Problem 2: transfers | Ward geometry, travel times | Transfers only within the city pool, ranked by spare capacity; no distances (the 60 ward IDs are synthetic and cannot be placed on real boundaries) |
| Problem 3: intervention effect | Measured larvicide/fogging efficacy | Configurable effect size with an explicit range, applied through the forecast model (counterfactual), reported as a scenario, not a measured effect |
| All | Ward population (only density per km²) | Prioritisation uses density and risk as the brief specifies; no per-capita rates |
| All | Delivery lead times, costs | Not modelled |

## 8. Saved results

`reports/onsite/resource_rule_identification.json` records the analysis; `app_data/resource_rules.json` is the runtime file the allocation engine reads: the rule identified on 2016–2018 only, which reproduces every later season.

In [14]:
out = {
    "dataset": {"file": "data/raw/onsite/bangladesh_dengue_large_dataset_onsite.csv", "sha256": ONSITE_SHA256,
                "development_rows": int(len(onsite)), "sealed_2025_rows": len(sealed)},
    "identification": {"window": "2016-01-04..2024-12-30 (development rows only)", "rule": "resource = floor(multiplier * cases + 0.5)",
                       "past_only_check": "rule identified on 2016-2018 reproduces 2019-2024 exactly"},
    "resources": {r: {"multiplier": float(v["a"]), "feasible_interval": [float(v["lo"]), float(v["hi"])],
                      "reproduction_rate": repro[r], "reproduction_2019_2024_from_2016_2018_rule": past_only[r]} for r, v in rule.items()},
    "replay_evaluation_2019_2024": ev.round(4).to_dict(orient="records"),
}
early_iv = {r: feasible(early, r) for r in RESOURCES}
runtime = {"rule": "resource = floor(multiplier * cases + 0.5)",
           "identified_on": "2016-01-04..2018-12-31: training-only seasons, before every replay origin (2019+), so no evaluated season informs it",
           "dataset_sha256": ONSITE_SHA256,
           "resources": {r: {"multiplier": float(sum(early_iv[r]) / 2), "feasible_interval": [float(early_iv[r][0]), float(early_iv[r][1])]} for r in RESOURCES}}
(ROOT / "app_data" / "resource_rules.json").write_text(json.dumps(runtime, indent=2) + "\n")
(ROOT / "reports" / "onsite").mkdir(parents=True, exist_ok=True)
(ROOT / "reports" / "onsite" / "resource_rule_identification.json").write_text(json.dumps(out, indent=2) + "\n")
print(json.dumps(out["resources"], indent=1))

{
 "saline_bags_needed": {
  "multiplier": 2.3848484848484848,
  "feasible_interval": [
   2.3833333333333333,
   2.3863636363636362
  ],
  "reproduction_rate": 1.0,
  "reproduction_2019_2024_from_2016_2018_rule": 1.0
 },
 "diagnostic_kits_needed": {
  "multiplier": 1.1642156862745099,
  "feasible_interval": [
   1.161764705882353,
   1.1666666666666667
  ],
  "reproduction_rate": 1.0,
  "reproduction_2019_2024_from_2016_2018_rule": 1.0
 },
 "hospital_beds_required": {
  "multiplier": 0.2984375,
  "feasible_interval": [
   0.296875,
   0.3
  ],
  "reproduction_rate": 1.0,
  "reproduction_2019_2024_from_2016_2018_rule": 1.0
 }
}
